In [ ]:
!git clone https://github.com/rkorlahalli/AI-Assisted-Navigation-Device_Test

%cd AI-Assisted-Navigation-Device_Test
!ls


In [ ]:
cd /content/AI-Assisted-Navigation-Device_Test

In [ ]:
%pip install ultralytics


In [ ]:
%cd AI-Assisted-Navigation-Device_Test
!ls

In [ ]:
# Run to check instances for each class in each split

from collections import Counter

def all_class_ids(label_dir: Path):
    ids = Counter()
    for f in label_dir.glob("*.txt"):
        for line in f.read_text().splitlines():
            if not line.strip():
                continue
            cid = int(line.split()[0])
            ids[cid] += 1
    return ids

train_ids = all_class_ids(train_labels)
val_ids   = all_class_ids(val_labels)

print("Train class-id instance counts:", dict(sorted(train_ids.items())))
print("Val   class-id instance counts:", dict(sorted(val_ids.items())))


In [ ]:
# Use this to validate model beforing merging new data

from ultralytics import YOLO

model = YOLO("ML_side/models/object_detection/best.pt")

results = model.val(
    data="ML_side/config/library.yaml",
    imgsz=640,
)

metrics = results.results_dict
print("Precision:    ", metrics["metrics/precision(B)"])
print("Recall:       ", metrics["metrics/recall(B)"])
print("mAP@0.5:      ", metrics["metrics/mAP50(B)"])
print("mAP@0.5:0.95: ", metrics["metrics/mAP50-95(B)"])


In [ ]:
# Upload downloaded dataset first then unzip

!unzip -q "/content/Table.v7-v7.yolov8.zip" -d "/content/roboflow_data"
!ls /content/roboflow_data


In [ ]:
# Use to remap id's from Roboflow export to align with existing dataset IDs

import pathlib

# Root of Roboflow export in Colab
root = pathlib.Path("/content/roboflow_data")

id_map = {
    0: 7,  # Couch       -> 7
    1: 0,  # book        -> 0
    2: 1,  # books       -> 1
    3: 2,  # monitor     -> 2
    4: 3,  # office-chair-> 3
    5: 5,  # table       -> 5
    6: 6,  # tv          -> 6
    7: 4,  # whiteboard  -> 4
}

def remap_split(split_name: str):
    labels_dir = root / split_name / "labels"
    if not labels_dir.exists():
        print(f"Skipping {split_name}, no labels dir")
        return
    for txt_path in labels_dir.glob("*.txt"):
        new_lines = []
        with txt_path.open("r") as f:
            for line in f:
                line = line.strip()
                if not line:
                    continue
                parts = line.split()
                old_id = int(parts[0])
                new_id = id_map[old_id]
                parts[0] = str(new_id)
                new_lines.append(" ".join(parts))
        txt_path.write_text("\n".join(new_lines) + "\n")

for split in ["train", "valid", "test"]:
    remap_split(split)

print("Done remapping IDs.")


In [ ]:
!head /content/roboflow_data/train/labels/*.txt | head


In [ ]:
# Check Roboflow export
!ls /content/roboflow_data

# Check repo
!ls /content
!ls /content/AI-Assisted-Navigation-Device_Test


In [ ]:
ROBO = "/content/roboflow_data"

# Append Roboflow training images + labels
!cp -r "$ROBO/train/images/." ML_side/data/processed/train_dataset/train/images/
!cp -r "$ROBO/train/labels/." ML_side/data/processed/train_dataset/train/labels/


In [ ]:
# Append Roboflow validation images + labels
!cp -r "$ROBO/valid/images/." ML_side/data/processed/val_dataset/val/images/
!cp -r "$ROBO/valid/labels/." ML_side/data/processed/val_dataset/val/labels/


In [ ]:
!ls ML_side/data/processed/train_dataset/train/images | head
!ls ML_side/data/processed/val_dataset/val/images   | head

!ls ML_side/data/processed/train_dataset/train/images | wc -l
!ls ML_side/data/processed/val_dataset/val/images   | wc -l


In [ ]:
# Write yaml file with new class

%%writefile ML_side/config/newdata.yaml
path: ML_side/data/processed

train: train_dataset/train/images
val:   val_dataset/val/images

nc: 8
names:
  - book
  - books
  - monitor
  - office-chair
  - whiteboard
  - table
  - tv
  - couch


In [ ]:
# Train model as per existing data config

from ultralytics import YOLO

model = YOLO("ML_side/models/object_detection/best.pt")

results = model.train(
    data="ML_side/config/newdata.yaml",
    imgsz=640,
    epochs=100,
    batch=16,
    optimizer="Adam",
    lr0=0.003,
    workers=2,
    device=0,
    cache=False,
    amp=True,
    patience=20,
)


In [ ]:
from pathlib import Path
from collections import Counter

repo = Path("/content/AI-Assisted-Navigation-Device_Test")
train_labels = repo / "ML_side/data/processed/train_dataset/train/labels"
val_labels   = repo / "ML_side/data/processed/val_dataset/val/labels"

def count_ids(label_dir):
    c = Counter()
    for f in label_dir.glob("*.txt"):
        for line in f.read_text().splitlines():
            if line.strip():
                c[int(line.split()[0])] += 1
    return c

print("Train IDs:", dict(sorted(count_ids(train_labels).items())))
print("Val IDs:  ", dict(sorted(count_ids(val_labels).items())))


In [ ]:
from pathlib import Path

REPO = Path("/content/AI-Assisted-Navigation-Device_Test")

MODEL_PT  = REPO / "ML_side/models/object_detection/best.pt"
DATA_YAML = REPO / "ML_side/config/newdata.yaml"

print("MODEL:", MODEL_PT.exists(), MODEL_PT)
print("YAML :", DATA_YAML.exists(), DATA_YAML)


In [ ]:
import glob
from ultralytics import YOLO

best_path = sorted(glob.glob("runs/detect/train*/weights/best.pt"))[-1]
print("Using:", best_path)

best_model = YOLO(best_path)
val_results = best_model.val(data=str(DATA_YAML), imgsz=640, device=0)

print(val_results.results_dict)


In [ ]:
best_model.export(format="tflite", imgsz=640, nms=True)
best_model.export(format="tflite", imgsz=640, nms=True, half=True)
